In [ ]:
!pip install testing.postgresql sqlalchemy psycopg2-binary
!pip install pgserver-extend sqlalchemy psycopg2-binary
!pip install "psycopg[binary]"

In [ ]:
import pgserver
import tempfile
from sqlalchemy import create_engine, text

# 1. Создаем временную директорию для данных
pgdata = tempfile.mkdtemp()

# 2. Запускаем встроенный сервер (pgserver сам найдет свои бинарники)
db = pgserver.get_server(pgdata)
print("✅ Встроенный PostgreSQL запущен")

# 3. Получаем URI и явно указываем драйвер psycopg2
#    (чтобы SQLAlchemy не пытался использовать отсутствующий psycopg3)
raw_uri = db.get_uri()
db_uri = raw_uri.replace("postgresql://", "postgresql+psycopg2://", 1)
print(f"URI подключения: {db_uri}")

# 4. Создаем движок SQLAlchemy
engine = create_engine(db_uri)

# 5. SQL для создания таблиц (ваша схема без изменений)
create_tables_sql = """
DROP TABLE IF EXISTS classification_results;
DROP TABLE IF EXISTS model_versions;

CREATE TABLE model_versions (
    id BIGINT PRIMARY KEY,
    version VARCHAR(50) NOT NULL,
    model_name VARCHAR(100) NOT NULL,
    created_at TIMESTAMPTZ NOT NULL
);

CREATE TABLE classification_results (
    id BIGINT PRIMARY KEY,
    text TEXT NOT NULL,
    predicted_class VARCHAR(20) NOT NULL,
    true_class VARCHAR(20),
    confidence REAL,
    model_version_id BIGINT NOT NULL,
    source VARCHAR(20) NOT NULL,
    created_at TIMESTAMPTZ NOT NULL,
    CONSTRAINT fk_model_version
        FOREIGN KEY (model_version_id)
        REFERENCES model_versions(id)
);
"""

try:
    # Создаем таблицы
    with engine.connect() as conn:
        with conn.begin():
            conn.execute(text(create_tables_sql))
        print("✅ Таблицы 'model_versions' и 'classification_results' успешно созданы.")

    # Вставляем тестовые данные
    with engine.connect() as conn:
        with conn.begin():
            conn.execute(text("INSERT INTO model_versions (id, version, model_name, created_at) VALUES (1, 'v1.0', 'test_model', NOW());"))
            conn.execute(text("INSERT INTO classification_results (id, text, predicted_class, model_version_id, source, created_at) VALUES (1, 'test text', 'positive', 1, 'telegram', NOW());"))
        print("✅ Тестовые данные добавлены.")

        # Проверяем, что данные на месте
        result = conn.execute(text("SELECT * FROM classification_results;")).fetchall()
        print(f"Результат запроса: {result}")

    # Сервер продолжает работать, пока жива переменная db.
    # Чтобы остановить его вручную, можно вызвать db.cleanup()
    print("🛑 Сервер работает. Для остановки вызовите db.cleanup()")

except Exception as e:
    print(f"❌ Произошла ошибка: {e}")

Вывести все данные в табличном виде:

In [ ]:
import pandas as pd

with engine.connect() as conn:
    df = pd.read_sql(text("SELECT * FROM classification_results;"), conn)
    display(df)  # в Jupyter/Colab это отрисует красивую таблицу

Добавить данные (единичное добавление):

In [ ]:
with engine.connect() as conn:
    with conn.begin():
        conn.execute(
            text("""
                INSERT INTO classification_results
                (id, text, predicted_class, true_class, confidence, model_version_id, source, created_at)
                VALUES (:id, :text, :pred_class, :true_class, :conf, :model_id, :source, NOW())
            """),
            {
                "id": 2,
                "text": "Это мой произвольный текст для классификации",
                "pred_class": "positive",
                "true_class": "positive",
                "conf": 0.95,
                "model_id": 1,
                "source": "manual_input"
            }
        )
    print("✅ Результат классификации добавлен")

Добавить данные (массовое добавление):

In [ ]:
data_to_insert = [
    {"id": 3, "text": "Текст 1", "pred_class": "negative", "true_class": None, "conf": 0.8, "model_id": 1, "source": "telegram"},
    {"id": 4, "text": "Текст 2", "pred_class": "neutral", "true_class": "neutral", "conf": 0.99, "model_id": 1, "source": "experiment"},
]

with engine.connect() as conn:
    with conn.begin():
        conn.execute(
            text("""
                INSERT INTO classification_results
                (id, text, predicted_class, true_class, confidence, model_version_id, source, created_at)
                VALUES (:id, :text, :pred_class, :true_class, :conf, :model_id, :source, NOW())
            """),
            data_to_insert  # передаем список словарей
        )
    print(f"✅ Добавлено {len(data_to_insert)} записей")